# Fine-tune Qwen3-0.6B as the incident-form extractor

**Run:** Runtime → Change runtime type → **GPU** (T4 works; A100 is ~3× faster) → *Run all*.
Upload **`laya_qwen_colab.zip`** when asked. At the end three files download:
`qwen3-incident-q8_0.gguf` (for Ollama on the PC), `qwen3-incident-hf.zip` (merged weights, for the Qualcomm
AI Hub / Genie compile for the 8 Elite) and `qwen_ft_report.json` (metrics). Send them back.

**What it does**
1. Builds extraction prompts from our dataset *exactly* as the live assistant sends them (`llm.build_prompt`).
2. A teacher (Qwen3-4B) fills in the **values** of fields our labels say were mentioned; the production
   validator keeps only allowed, grounded values.
3. LoRA-fine-tunes **Qwen3-0.6B** (thinking off, loss on the JSON answer only), picks the best epoch on validation.
4. Scores base vs fine-tuned Qwen3-0.6B on the held-out test conversations; exports merged HF + GGUF.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "transformers>=4.51" peft accelerate

In [ ]:
import os, sys, zipfile
from google.colab import files
if not os.path.exists("laya_qwen_colab.zip"):
    files.upload()
zipfile.ZipFile("laya_qwen_colab.zip").extractall("work")
os.chdir("work"); sys.path.insert(0, "laya")
import json, math, time, random, copy, gc
import torch
import qwen_ft_lib as Q, llm as L, form as F
from transformers import AutoTokenizer, AutoModelForCausalLM
TEACHER = "Qwen/Qwen3-4B"        # bigger teacher if you have an A100: "Qwen/Qwen3-8B"
STUDENT = "Qwen/Qwen3-0.6B"
dev = "cuda"
DT = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
E = Q.build_examples("datagen/out")
print({k: len(v) for k, v in E.items()}, "| dtype", DT)

In [ ]:
# ---- 1. teacher labels: values for the fields our labels say were mentioned
t0 = time.time()
tt = AutoTokenizer.from_pretrained(TEACHER); tt.padding_side = "left"
tm = AutoModelForCausalLM.from_pretrained(TEACHER, torch_dtype=DT, device_map="cuda").eval()

@torch.no_grad()
def teach(batch):
    texts = [tt.apply_chat_template(Q.teacher_messages(e), tokenize=False, add_generation_prompt=True, enable_thinking=False) for e in batch]
    enc = tt(texts, return_tensors="pt", padding=True).to(dev)
    out = tm.generate(**enc, max_new_tokens=96, do_sample=False, pad_token_id=tt.pad_token_id)
    return tt.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

D = {}
for split, exs in E.items():
    done, need = [], [e for e in exs if e["mentioned"]]
    raw = {}
    for i in range(0, len(need), 32):
        b = need[i:i + 32]
        for e, r in zip(b, teach(b)):
            raw[id(e)] = r
    for e in exs:
        f = Q.finalize(e, raw.get(id(e), "{}"))
        if f: done.append(f)
    D[split] = done
    print(split, len(done), "examples |", sum(len(x["dropped"]) for x in done), "teacher values rejected by the validator |",
          f"{time.time() - t0:.0f}s")
json.dump(D, open("/content/extract_data.json", "w"))
print("example:", json.dumps(D["train"][3])[:300])
del tm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 2. student: tokenizer, prompt budget, base-model score on test
st = AutoTokenizer.from_pretrained(STUDENT); st.padding_side = "left"
END = st.convert_tokens_to_ids("<|im_end|>")
plen = [len(st(Q.student_text(e, False), add_special_tokens=False).input_ids) for e in D["test"]]
tot = [len(st(Q.student_text(e, True), add_special_tokens=False).input_ids) for e in D["test"]]
print(f"prompt tokens median {sorted(plen)[len(plen)//2]} max {max(plen)} | prompt+answer <= 256: {sum(t <= 256 for t in tot)/len(tot):.1%}")

@torch.no_grad()
def generate(model, exs, bs=32):
    model.eval(); outs = []
    for i in range(0, len(exs), bs):
        enc = st([Q.student_text(e, False) for e in exs[i:i + bs]], return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
        o = model.generate(**enc, max_new_tokens=64, do_sample=False, eos_token_id=END, pad_token_id=st.pad_token_id)
        outs += st.batch_decode(o[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

base = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=DT, device_map="cuda")
base_test = Q.score(D["test"], generate(base, D["test"]))
print("BASE Qwen3-0.6B:", json.dumps(base_test, indent=1))
del base; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 3. LoRA fine-tune (loss on the JSON answer only), best epoch on validation
from peft import LoraConfig, get_peft_model
EPOCHS, BS, LR, R = 3, 16, 2e-4, 16
random.seed(0); torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=torch.float32, device_map="cuda")
model = get_peft_model(model, LoraConfig(r=R, lora_alpha=2 * R, lora_dropout=0.05, task_type="CAUSAL_LM",
                       target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def encode(e):
    p = st(Q.student_text(e, False), add_special_tokens=False).input_ids
    full = st(Q.student_text(e, True), add_special_tokens=False).input_ids
    return full, [-100] * len(p) + full[len(p):]

train = [encode(e) for e in D["train"]]
def batches():
    idx = list(range(len(train))); random.shuffle(idx)
    for i in range(0, len(idx), BS):
        b = [train[j] for j in idx[i:i + BS]]; n = max(len(x[0]) for x in b)
        ids = torch.tensor([x[0] + [st.pad_token_id] * (n - len(x[0])) for x in b])
        lab = torch.tensor([x[1] + [-100] * (n - len(x[1])) for x in b])
        att = torch.tensor([[1] * len(x[0]) + [0] * (n - len(x[0])) for x in b])
        yield ids.to(dev), att.to(dev), lab.to(dev)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
steps = EPOCHS * math.ceil(len(train) / BS)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, s / (0.05 * steps)) * 0.5 * (1 + math.cos(math.pi * min(1, s / steps))))
scaler = torch.amp.GradScaler("cuda", enabled=DT == torch.float16)
val_sub = D["val"][:150]
best, best_state, t0 = -1, None, time.time()
for ep in range(EPOCHS):
    model.train(); tot = n = 0
    for ids, att, lab in batches():
        with torch.autocast("cuda", dtype=DT):
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(opt); scaler.update(); sched.step(); tot += float(loss.detach()); n += 1
    v = Q.score(val_sub, generate(model, val_sub))
    print(f"epoch {ep + 1}: loss {tot / n:.3f} | val exact {v['exact_match']:.3f} field F1 {v['field_f1']:.3f} json {v['json_valid']:.2f} | {time.time() - t0:.0f}s")
    if v["field_f1"] > best:
        best, best_state = v["field_f1"], {k: x.detach().cpu().clone() for k, x in model.state_dict().items() if "lora_" in k}
model.load_state_dict(best_state, strict=False)
ft_test = Q.score(D["test"], generate(model, D["test"]))
print("FINE-TUNED Qwen3-0.6B:", json.dumps(ft_test, indent=1))
print(f"\n{'':22s}{'base':>10s}{'fine-tuned':>12s}")
for k in ("json_valid", "field_precision", "field_recall", "field_f1", "exact_match"):
    print(f"{k:22s}{base_test[k]:10.3f}{ft_test[k]:12.3f}")

In [ ]:
# ---- 4. export: merged HF weights (for AI Hub / Genie on the 8 Elite) + GGUF q8_0 (for Ollama on the PC)
merged = model.merge_and_unload().to(torch.float16)
merged.save_pretrained("/content/qwen3-incident", safe_serialization=True); st.save_pretrained("/content/qwen3-incident")
!cd /content && zip -qr qwen3-incident-hf.zip qwen3-incident && ls -la qwen3-incident-hf.zip
!cd /content && [ -d llama.cpp ] || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!cd /content && python llama.cpp/convert_hf_to_gguf.py qwen3-incident --outtype q8_0 --outfile qwen3-incident-q8_0.gguf 2>&1 | tail -2
report = {"teacher": TEACHER, "student": STUDENT, "examples": {k: len(v) for k, v in D.items()},
          "base_test": base_test, "finetuned_test": ft_test, "epochs": EPOCHS, "lora_r": R, "lr": LR,
          "gpu": torch.cuda.get_device_name(), "prompt_tokens_max": max(plen)}
json.dump(report, open("/content/qwen_ft_report.json", "w"), indent=1)
for f in ("qwen_ft_report.json", "qwen3-incident-q8_0.gguf", "qwen3-incident-hf.zip"):
    files.download("/content/" + f)